# 01. EDA — 대회 시작 직후 데이터 파악
> **여기만 바꾸면 된다** → `CONFIG` 블록의 경로·컬럼명
> 순서대로 실행하면 30분 안에 데이터 파악 완료

In [ ]:
# =============================================
# ★ 여기만 대회에 맞게 수정 ★
# =============================================
CONFIG = dict(
    train_csv   = 'input/train_metadata.csv',
    audio_dir   = 'input/train_audio',
    label_col   = 'primary_label',   # 라벨 컬럼명
    id_col      = 'filename',         # 파일 id 컬럼명
    sample_rate = 32000,
    n_samples   = 8,                  # 시각화할 샘플 수
    silence_db  = -50,                # 무음 판정 dB 기준
)
# =============================================

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
print('라이브러리 로드 완료')

## 1. 메타데이터 기본 파악

In [ ]:
df = pd.read_csv(CONFIG['train_csv'])
print(f'행 수: {len(df):,} | 열 수: {df.shape[1]}')
print(f'컬럼: {list(df.columns)}')
df.head()

In [ ]:
print('=== 기본 정보 ===')
print(df.dtypes)
print()
print('=== 결측값 ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print(f'=== 라벨 컬럼: [{CONFIG["label_col"]}] ===')
print(f'고유 클래스 수: {df[CONFIG["label_col"]].nunique()}')
print(f'multilabel 여부(,포함): {df[CONFIG["label_col"]].astype(str).str.contains(",").any()}')

## 2. 클래스 분포 (불균형 확인)

In [ ]:
label_col = CONFIG['label_col']
counts = df[label_col].value_counts()
n_classes = len(counts)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 전체 분포
axes[0].bar(range(n_classes), sorted(counts.values, reverse=True), color='steelblue', alpha=0.7)
axes[0].set_title(f'클래스별 샘플 수 (전체 {n_classes}개 클래스)')
axes[0].set_xlabel('클래스 인덱스 (샘플 수 내림차순)')
axes[0].set_ylabel('샘플 수')

# 샘플 수 분포 histogram
axes[1].hist(counts.values, bins=30, color='coral', alpha=0.7, edgecolor='white')
axes[1].set_title('클래스당 샘플 수 분포')
axes[1].set_xlabel('샘플 수')
axes[1].axvline(counts.median(), color='red', linestyle='--', label=f'중앙값: {counts.median():.0f}')
axes[1].legend()

# Top / Bottom 10
top10 = counts.head(10)
axes[2].barh(range(10), top10.values[::-1], color='steelblue', alpha=0.7)
axes[2].set_yticks(range(10))
axes[2].set_yticklabels(top10.index[::-1], fontsize=8)
axes[2].set_title('샘플 수 Top 10 클래스')

plt.tight_layout()
plt.show()

print(f'최다 클래스: {counts.index[0]} ({counts.iloc[0]}개)')
print(f'최소 클래스: {counts.index[-1]} ({counts.iloc[-1]}개)')
print(f'불균형 비율 (max/min): {counts.iloc[0]/counts.iloc[-1]:.1f}x')
print(f'샘플 10개 미만 클래스: {(counts < 10).sum()}개')
print(f'샘플 50개 미만 클래스: {(counts < 50).sum()}개')
print()
print('--- 전략 힌트 ---')
if counts.iloc[0]/counts.iloc[-1] > 20:
    print('⚠ 불균형 심함 → Focal Loss 또는 class_weights 고려')
elif counts.iloc[0]/counts.iloc[-1] > 5:
    print('△ 불균형 보통 → label_smoothing + mixup으로 대응 가능')
else:
    print('✓ 균형 양호 → 일반 BCE 사용 가능')

## 3. Duration 분포 (crop 전략 결정)

In [ ]:
# duration 컬럼이 없으면 직접 계산
if 'duration' not in df.columns:
    print('duration 컬럼 없음 → 샘플 500개로 추정 중...')
    durations = []
    sample_ids = df[CONFIG['id_col']].sample(min(500, len(df)), random_state=42)
    for sid in tqdm(sample_ids):
        path = Path(CONFIG['audio_dir']) / sid
        try:
            d = librosa.get_duration(path=path)
            durations.append(d)
        except:
            pass
    dur_series = pd.Series(durations)
else:
    dur_series = df['duration']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(dur_series, bins=50, color='mediumseagreen', alpha=0.7, edgecolor='white')
axes[0].set_title('Duration 분포')
axes[0].set_xlabel('초(sec)')
axes[0].axvline(dur_series.mean(), color='red', linestyle='--', label=f'평균: {dur_series.mean():.1f}s')
axes[0].axvline(dur_series.median(), color='orange', linestyle='--', label=f'중앙값: {dur_series.median():.1f}s')
axes[0].legend()

axes[1].hist(dur_series.clip(upper=60), bins=50, color='slateblue', alpha=0.7, edgecolor='white')
axes[1].set_title('Duration 분포 (60초 clip)')
axes[1].set_xlabel('초(sec)')

plt.tight_layout(); plt.show()

print(f'평균: {dur_series.mean():.1f}s | 중앙값: {dur_series.median():.1f}s')
print(f'최소: {dur_series.min():.1f}s | 최대: {dur_series.max():.1f}s')
print(f'5초 미만: {(dur_series < 5).sum()}개 ({(dur_series<5).mean()*100:.1f}%)')
print(f'30초 초과: {(dur_series > 30).sum()}개 ({(dur_series>30).mean()*100:.1f}%)')
print()
print('--- 전략 힌트 ---')
if dur_series.median() > 30:
    print('⚠ 긴 오디오(soundscape) → sliding window 추론 필수')
elif dur_series.median() > 10:
    print('△ 중간 길이 → random crop으로 5~10초 클립 사용')
else:
    print('✓ 짧은 클립 → 그대로 사용 or pad')

## 4. 오디오 샘플 시각화 (파형 + Mel-spectrogram)

In [ ]:
def plot_audio_sample(path, label, sr=32000, ax_wave=None, ax_mel=None):
    try:
        wav, _ = librosa.load(path, sr=sr, duration=10.0)
        mel = librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=128, fmax=sr//2)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        if ax_wave:
            ax_wave.plot(np.linspace(0, len(wav)/sr, len(wav)), wav, color='steelblue', lw=0.5)
            ax_wave.set_title(f'{label}\n파형', fontsize=8)
            ax_wave.set_xlabel('초'); ax_wave.set_yticks([])
        if ax_mel:
            librosa.display.specshow(mel_db, sr=sr, hop_length=512,
                                     x_axis='time', y_axis='mel', ax=ax_mel)
            ax_mel.set_title(f'{label}\nMel-spectrogram', fontsize=8)
    except Exception as e:
        print(f'로드 실패: {path} ({e})')

# 클래스별 1개씩 샘플
n = CONFIG['n_samples']
sample_classes = df[CONFIG['label_col']].value_counts().index[:n].tolist()
fig, axes = plt.subplots(n, 2, figsize=(14, n*2.5))

for i, cls in enumerate(sample_classes):
    row = df[df[CONFIG['label_col']] == cls].iloc[0]
    path = Path(CONFIG['audio_dir']) / row[CONFIG['id_col']]
    plot_audio_sample(path, cls, CONFIG['sample_rate'],
                      ax_wave=axes[i, 0], ax_mel=axes[i, 1])

plt.tight_layout(); plt.show()

## 5. Sample Rate 검증

In [ ]:
print('샘플 100개로 SR 분포 확인 중...')
sr_list = []
for sid in tqdm(df[CONFIG['id_col']].sample(min(100, len(df)), random_state=42)):
    path = Path(CONFIG['audio_dir']) / sid
    try:
        sr = librosa.get_samplerate(str(path))
        sr_list.append(sr)
    except:
        sr_list.append(None)

sr_series = pd.Series(sr_list)
print(sr_series.value_counts())
if sr_series.nunique() > 1:
    print('⚠ SR이 통일되지 않음 → preprocessing.py에서 리샘플 필수')
else:
    print(f'✓ 전부 {sr_series.iloc[0]}Hz')
    print(f'  config sample_rate: {CONFIG["sample_rate"]} → ', end='')
    print('일치 ✓' if sr_series.iloc[0] == CONFIG['sample_rate'] else f'불일치 ⚠ config를 {sr_series.iloc[0]}으로 변경 권장')

## 6. 무음 / 저에너지 샘플 탐지

In [ ]:
print('샘플 200개로 RMS 에너지 분포 확인 중...')
rms_list = []
ids = df[CONFIG['id_col']].sample(min(200, len(df)), random_state=42).tolist()
for sid in tqdm(ids):
    path = Path(CONFIG['audio_dir']) / sid
    try:
        wav, _ = librosa.load(path, sr=CONFIG['sample_rate'], duration=5.0)
        rms = 20 * np.log10(np.sqrt(np.mean(wav**2)) + 1e-9)
        rms_list.append({'id': sid, 'rms_db': rms})
    except:
        rms_list.append({'id': sid, 'rms_db': np.nan})

rms_df = pd.DataFrame(rms_list)

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(rms_df['rms_db'].dropna(), bins=40, color='coral', alpha=0.7, edgecolor='white')
ax.axvline(CONFIG['silence_db'], color='red', linestyle='--', label=f'무음 기준: {CONFIG["silence_db"]}dB')
ax.set_title('RMS 에너지 분포 (dB)')
ax.set_xlabel('RMS (dB)'); ax.legend()
plt.tight_layout(); plt.show()

silence_mask = rms_df['rms_db'] < CONFIG['silence_db']
print(f'무음 의심 샘플: {silence_mask.sum()}개 ({silence_mask.mean()*100:.1f}%)')
if silence_mask.sum() > 0:
    print('무음 의심 파일 목록:')
    print(rms_df[silence_mask][['id','rms_db']].to_string())
    print('\n→ verify_audio.py 실행 권장')

## 7. EDA 요약 → 전략 결정

In [ ]:
print('='*50)
print('EDA 요약')
print('='*50)
print(f'클래스 수       : {df[CONFIG["label_col"]].nunique()}')
print(f'총 샘플         : {len(df):,}')
print(f'클래스당 중앙값 : {df[CONFIG["label_col"]].value_counts().median():.0f}개')
print(f'불균형 비율     : {counts.iloc[0]/counts.iloc[-1]:.1f}x')
print()
print('--- config 업데이트 체크리스트 ---')
print(f'data.label_col   = "{CONFIG["label_col"]}"')
print(f'data.id_col      = "{CONFIG["id_col"]}"')
print(f'data.sample_rate = {CONFIG["sample_rate"]}')
print(f'model.num_classes= {df[CONFIG["label_col"]].nunique()}')
print('='*50)